<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.es/cap03/cap03.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Práctica con Ejercicios de Programación**

### 🎯 Objetivo de este Cuaderno

El cuaderno permite desarrollar, validar, organizar y probar soluciones de **Ejercicios de Programación (EPs)** en entornos interactivos, como Colab, con los mismos casos de prueba de Moodle, copiándolos allí solo al momento de registrar la nota oficial.

#### *Download*

Descargue `morph.py` y `testsuite.py` ejecutando la celda a continuación:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Ejecutando los Tests
Para evaluar los tests, ejecuta `TestSuite("EP03_01.extensión").run()` en una nueva celda, reemplazando la extensión por la del lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema descarga los casos de prueba de GitHub, ejecuta el programa y calcula la nota automáticamente.

Para probar código Python directamente, sin guardar un archivo, usa `run_code(codigo)` pasando el código como *string* en una variable `codigo`:

```python
codigo = """
from morph import mm
# ... tu código aquí ...
"""
TestSuite("EP03_01").run_code(codigo)
```

### EP03_01 ➕ Adición Saturada de Constante

En sistemas de vigilancia por video, las cámaras en entornos con iluminación variable producen imágenes subexpuestas. El ajuste de brillo mediante **adición saturada de una constante** es la operación más simple para la corrección inmediata, aplicándose en tiempo real en los *chips* de cámaras embebidas y en *pipelines* de preprocesamiento de robots móviles.

Ver en [Figura 3.1](#fig-03-sim-ep0301-adicao) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Constante:** Leer el entero $k$ (valor a sumar).
3. **Datos:** Leer los valores enteros de la matriz original fila por fila.
4. **Mapeo:** Para cada píxel $p$, calcular el nuevo valor mediante la ecuación:

$$p' = \text{clip}(p + k)$$

5. **Salida:** Mostrar la matriz resultante con dimensiones $L \times C$.

#### 📌 Restricciones Computacionales

* **Saturación (*Clipping*):** Los valores deben confinarse al intervalo $[0, 255]$:
$$\text{clip}(x) = \max(0, \min(255, x))$$
* **Tipo:** El resultado final debe ser entero (sin decimales).
* **$k$ puede ser negativo:** los valores negativos oscurecen la imagen; los positivos la aclaran.

#### 🧠 Fundamentación Teórica

| Parámetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$k > 0$** | Entero | Aclara la imagen; los píxeles cercanos a 255 saturan en blanco |
| **$k < 0$** | Entero | Oscurece la imagen; los píxeles cercanos a 0 saturan en negro |
| **$k = 0$** | Entero | Imagen sin cambios |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $k$.
* Líneas siguientes: Elementos enteros de la matriz original.

**Salida:**

* Matriz transformada en $L$ filas y $C$ columnas, valores enteros separados por espacios.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 2<br>3<br>50<br>0 100 200<br>210 240 255 | 50 150 250<br>255 255 255 | Saturación en 255 en los píxeles altos |
| 1<br>4<br>-30<br>0 20 200 255 | 0 0 170 225 | Saturación en 0 en los píxeles bajos |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0301-adicao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">➕ Simulador EP03_01: Adición Saturada de Constante</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = clip(p + k)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajuste el valor de la constante k para observar el desplazamiento de brillo de la imagen y el truncamiento por saturación en el intervalo [0, 255].</p>

    <!-- Controle da Constante k -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#27ae60;">Constante (k)</label>
        <span id="sim_ep0301_vl_k" style="font-family:monospace;font-size:12px;font-weight:700;color:#27ae60;">0</span>
      </div>
      <input type="range" id="sim_ep0301_sl_k" min="-128" max="128" step="1" value="0" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (p)</span>
        <div id="sim_ep0301_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0301_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Resultado Transformado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Transformado (p')</span>
        <div id="sim_ep0301_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0301_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Restablecer (k = 0)</button>
      </div>

    </div>

    <!-- Mensagem Explicativa Dinâmica -->
    <div id="sim_ep0301_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula aplicada: <b>clip(p + (0))</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0301(root){
    if (!root || root.dataset.simEp0301Init) return;
    root.dataset.simEp0301Init = "1";

    var slK      = root.querySelector('#sim_ep0301_sl_k');
    var vlK      = root.querySelector('#sim_ep0301_vl_k');
    var gridOrig = root.querySelector('#sim_ep0301_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0301_grid_new');
    var debugDiv = root.querySelector('#sim_ep0301_debug');

    var btnNew   = root.querySelector('#sim_ep0301_btnNew');
    var btnReset = root.querySelector('#sim_ep0301_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function render() {
      var k = parseInt(slK.value) || 0;
      vlK.textContent = k;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>clip(p + (' + k + '))</b>';

      gridOrig.innerHTML = '';
      gridNew.innerHTML  = '';

      pixels.forEach(function(p) {
        // Célula Original
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gridOrig.appendChild(cellO);

        // Célula Resultado
        var res = Math.max(0, Math.min(255, p + k));
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slK.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slK.value = 0;
      render();
    });

    render();
  }

  function tryInitSimEP0301(){
    var root = document.getElementById('sim-ep0301-adicao');
    if (root) initSimEP0301(root); else setTimeout(tryInitSimEP0301, 200);
  }
  tryInitSimEP0301();
})();
</script>
</div>
""")

**Figura 3.1:** Simulador EP03_01: Adición Saturada de Constante (p


<figure id="fig-03-sim-ep0301-adicao">
  <img src="imagens/fig-03-sim-ep0301-adicao.png" alt=" Simulador EP03_01: Adición Saturada de Constante (p' = clip(p + k)) " style="max-width:80%" />
  <figcaption><strong>Figura 3.1:</strong>  Simulador EP03_01: Adición Saturada de Constante (p' = clip(p + k)) </figcaption>
</figure>

In [ ]:
%%writefile EP03_01.py
# Código Python

In [ ]:
TestSuite("EP03_01.py").run()

### EP03_02 🔀 Alpha *Blending* de Dos Imágenes

En medicina nuclear, imágenes de diferentes modalidades (tomografía computarizada y resonancia magnética) se fusionan para ayudar en el diagnóstico. La **mezcla ponderada** (*alpha blending*) es la operación fundamental de este proceso, permitiendo al radiólogo controlar interactivamente el peso de cada modalidad en la imagen mostrada.

Ver en [Figura 3.2](#fig-03-sim-ep0302-blending) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Parámetro:** Leer el valor real $\alpha \in [0, 1]$.
3. **Datos:** Leer los valores enteros de la matriz $f_1$ (imagen 1) y luego de la matriz $f_2$ (imagen 2).
4. **Mapeo:** Para cada posición $(i, j)$, calcular:

$$g(i,j) = \text{clip}\left(\text{round}\left(\alpha \cdot f_1(i,j) + (1-\alpha) \cdot f_2(i,j)\right)\right)$$

5. **Salida:** Mostrar la matriz resultante $L \times C$.

#### 📌 Restricciones Computacionales

* **Redondeo:** Aplicar `round` antes de la conversión a entero.
* **Saturación:** Confinar al intervalo $[0, 255]$ con $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operación en float:** Realizar la operación en punto flotante antes de redondear.

#### 🧠 Fundamentación Teórica

| Valor de $\alpha$ | Resultado |
|:-----------------:|:----------|
| $\alpha = 1.0$ | Solo $f_1$ |
| $\alpha = 0.5$ | Media aritmética de $f_1$ y $f_2$ |
| $\alpha = 0.0$ | Solo $f_2$ |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Real $\alpha$.
* Líneas siguientes: Elementos de $f_1$ ($L$ líneas con $C$ valores cada una).
* Líneas siguientes: Elementos de $f_2$ ($L$ líneas con $C$ valores cada una).

**Salida:**

* Matriz resultante $L \times C$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 1<br>3<br>0.5<br>0 100 200<br>100 200 50 | 50 150 125 | Media entre las dos imágenes |
| 1<br>3<br>1.0<br>10 20 30<br>90 80 70 | 10 20 30 | Solo $f_1$ (alpha=1) |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0302-blending" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🔀 Simulador EP03_02: Alpha Blending de Dos Imágenes</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = α·f1 + (1−α)·f2</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajusta el parámetro de transparencia α para observar la combinación lineal ponderada píxel a píxel entre las imágenes f1 y f2.</p>

    <!-- Controle do Parâmetro Alpha -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#8e44ad;">α (Alpha — Peso de f1)</label>
        <span id="sim_ep0302_vl_a" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;">0.50</span>
      </div>
      <input type="range" id="sim_ep0302_sl_a" min="0" max="1" step="0.05" value="0.5" style="width:100%;cursor:pointer;">
      <div style="margin-top:6px;font-size:10px;color:#8a8371;text-align:center;font-family:monospace;">
        α = 0.00 → Solo f2 &nbsp;|&nbsp; α = 0.50 → Media Ponderada Igual &nbsp;|&nbsp; α = 1.00 → Solo f1
      </div>
    </div>

    <!-- Comparativo em 3 Colunas: f1 vs f2 vs Resultado g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(160px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem f1 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen f1</span>
        <div id="sim_ep0302_grid_f1" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0302_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuevas Imágenes</button>
      </div>

      <!-- Imagem f2 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen f2</span>
        <div id="sim_ep0302_grid_f2" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado g -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado g</span>
        <div id="sim_ep0302_grid_g" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0302_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Restablecer (α = 0.5)</button>
      </div>

    </div>

    <!-- Mensagem Explicativa Dinâmica -->
    <div id="sim_ep0302_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula: <b>clip(round(0.50 · f1 + 0.50 · f2))</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0302(root){
    if (!root || root.dataset.simEp0302Init) return;
    root.dataset.simEp0302Init = "1";

    var slA      = root.querySelector('#sim_ep0302_sl_a');
    var vlA      = root.querySelector('#sim_ep0302_vl_a');
    var gF1      = root.querySelector('#sim_ep0302_grid_f1');
    var gF2      = root.querySelector('#sim_ep0302_grid_f2');
    var gG       = root.querySelector('#sim_ep0302_grid_g');
    var debugDiv = root.querySelector('#sim_ep0302_debug');

    var btnNew   = root.querySelector('#sim_ep0302_btnNew');
    var btnReset = root.querySelector('#sim_ep0302_btnReset');

    var px1 = [], px2 = [];

    function generate() {
      px1 = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
      px2 = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
    }

    function createCell(val) {
      var c = document.createElement('div');
      var fgColor = val > 128 ? '#000000' : '#ffffff';
      c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + val + ',' + val + ',' + val + ');color:' + fgColor + ';box-sizing:border-box;';
      c.textContent = val;
      return c;
    }

    function render() {
      var a = parseFloat(slA.value) || 0;
      var a1 = a.toFixed(2);
      var a2 = (1 - a).toFixed(2);

      vlA.textContent = a1;
      debugDiv.innerHTML = 'Fórmula: <b>clip(round(' + a1 + ' · f1 + ' + a2 + ' · f2))</b>';

      gF1.innerHTML = '';
      gF2.innerHTML = '';
      gG.innerHTML  = '';

      for (var i = 0; i < 16; i++) {
        gF1.appendChild(createCell(px1[i]));
        gF2.appendChild(createCell(px2[i]));

        var res = Math.max(0, Math.min(255, Math.round(a * px1[i] + (1 - a) * px2[i])));
        gG.appendChild(createCell(res));
      }
    }

    slA.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      generate();
      render();
    });

    btnReset.addEventListener('click', function() {
      slA.value = '0.5';
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0302(){
    var root = document.getElementById('sim-ep0302-blending');
    if (root) initSimEP0302(root); else setTimeout(tryInitSimEP0302, 200);
  }
  tryInitSimEP0302();
})();
</script>
</div>
""")

**Figura 3.2:** Simulador EP03_02: Mezcla alfa de dos imágenes (g = α·f1 + (1−α)·f2)


<figure id="fig-03-sim-ep0302-blending">
  <img src="imagens/fig-03-sim-ep0302-blending.png" alt=" Simulador EP03_02: Mezcla alfa de dos imágenes (g = α·f1 + (1−α)·f2) " style="max-width:80%" />
  <figcaption><strong>Figura 3.2:</strong>  Simulador EP03_02: Mezcla alfa de dos imágenes (g = α·f1 + (1−α)·f2) </figcaption>
</figure>

In [ ]:
%%writefile EP03_02.py
# Código Python

In [ ]:
TestSuite("EP03_02.py").run()

### EP03_03 🎭 Inversión de Imagen (Negativo Fotográfico)

En radiología, las imágenes de rayos X se visualizan tradicionalmente en negativo: los huesos aparecen en negro sobre fondo blanco. La operación de **negativo fotográfico** se aplica rutinariamente en PACS (*Picture Archiving and Communication Systems*) para facilitar la detección de fracturas y densidades óseas.

Ver en [Figura 3.3](#fig-03-sim-ep0303-inversao) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer los valores enteros de la matriz original.
3. **Mapeo:** Para cada píxel $p$, calcular el negativo:

$$p' = 255 - p$$

4. **Salida:** Mostrar la matriz resultante de $L \times C$.

#### 📌 Restricciones Computacionales

* **Sin necesidad de *recorte*:** El resultado de $255 - p$ con $p \in [0, 255]$ siempre está $\in [0, 255]$.
* **Tipo entero:** La salida debe ser valores enteros.
* **Equivalencia lógica:** La operación es idéntica al `NOT` bit a bit (`mm.bnot`) en imágenes de 8 bits.

#### 🧠 Fundamentación Teórica

| Píxel Original $p$ | Píxel Negativo $p'$ | Observación |
|:------------------:|:-------------------:|:----------:|
| 0 (negro) | 255 (blanco) | Inversión total |
| 128 (gris medio) | 127 (gris medio) | Valor central |
| 255 (blanco) | 0 (negro) | Inversión total |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos enteros de la matriz original.

**Salida:**

* Matriz negativa en $L$ filas y $C$ columnas.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 1<br>4<br>0 128 200 255 | 255 127 55 0 | Inversión de cada píxel |
| 2<br>2<br>10 20<br>30 40 | 245 235<br>225 215 | Matriz 2x2 invertida |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0303-inversao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎭 Simulador EP03_03: Negativo Fotográfico (Inversión)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = 255 − p</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Observe la inversión complementaria de intensidad: los tonos oscuros se vuelven claros y los tonos claros se vuelven oscuros restando cada píxel del valor máximo de 255.</p>

    <!-- Comparativo Lado a Lado: Entrada vs Negativo -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (p)</span>
        <div id="sim_ep0303_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0303_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Negativo (p' = 255 - p) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Negativo (p' = 255 − p)</span>
        <div id="sim_ep0303_grid_neg" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel de Informação Explicativo -->
    <div id="sim_ep0303_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula aplicada: <b>p' = 255 − p</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0303(root){
    if (!root || root.dataset.simEp0303Init) return;
    root.dataset.simEp0303Init = "1";

    var gO = root.querySelector('#sim_ep0303_grid_orig');
    var gN = root.querySelector('#sim_ep0303_grid_neg');
    var btnNew = root.querySelector('#sim_ep0303_btnNew');

    var pixels = [];

    function generate() {
      pixels = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
    }

    function render() {
      gO.innerHTML = '';
      gN.innerHTML = '';

      pixels.forEach(function(p) {
        // Célula Original
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gO.appendChild(cellO);

        // Célula Negativo
        var r = 255 - p;
        var fgColorN = r > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = r;
        gN.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function() {
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0303(){
    var root = document.getElementById('sim-ep0303-inversao');
    if (root) initSimEP0303(root); else setTimeout(tryInitSimEP0303, 200);
  }
  tryInitSimEP0303();
})();
</script>
""")

**Figura 3.3:** Simulador EP03_03: Inversión de Imagen — Negativo Fotográfico (p


<figure id="fig-03-sim-ep0303-inversao">
  <img src="imagens/fig-03-sim-ep0303-inversao.png" alt=" Simulador EP03_03: Inversión de Imagen — Negativo Fotográfico (p' = 255 − p) " style="max-width:80%" />
  <figcaption><strong>Figura 3.3:</strong>  Simulador EP03_03: Inversión de Imagen — Negativo Fotográfico (p' = 255 − p) </figcaption>
</figure>

In [ ]:
%%writefile EP03_03.py
# Código Python

In [ ]:
TestSuite("EP03_03.py").run()

### EP03_04 📊 Equalización de Histograma (L bits)

En imágenes de satélite de teleobservación, la variación de iluminación a lo largo del día produce imágenes de bajo contraste. La **equalización de histograma** se aplica automáticamente en satélites como el Landsat para redistribuir los tonos, revelando detalles de vegetación, relieve y zonas urbanas invisibles en la imagen original.

Ver en la [Figura 3.4](#fig-03-sim-ep0304-equalizacao) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (líneas), $C$ (columnas) y $B$ (número de bits, con $L_{\max} = 2^B$).
2. **Datos:** Leer la matriz de píxeles $f$ con valores en $[0, 2^B - 1]$.
3. **Histograma:** Calcular $h[k]$ = número de píxeles con intensidad $k$, para $k = 0 \ldots 2^B-1$.
4. **Probabilidad:** $p[k] = h[k] / (L \cdot C)$.
5. **CDF:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$; función de distribución acumulada.
6. **LUT:** $\text{lut}[k] = \text{round}\left(\text{cdf}[k] \cdot (2^B - 1)\right)$; *Look-Up Table* (tabla de consulta).
7. **Aplicación:** $g[i,j] = \text{lut}[f[i,j]]$.
8. **Salida:** Mostrar la matriz equalizada $L \times C$.

#### 📌 Restricciones Computacionales

* **Redondeo:** Usar redondeo matemático (`round`) en la LUT.
* **Bits:** El número de niveles es $2^B$ (ej.: $B=3 \Rightarrow 8$ niveles, $B=8 \Rightarrow 256$ niveles).
* **CDF acumulada:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$, con $\text{cdf}[2^B-1] = 1.0$.

#### 🧠 Fundamentación Teórica

| Etapa | Operación | Fórmula |
|:-----:|:---------|:--------|
| 1 | Histograma | $h[k] \leftarrow$ nº píxeles con intensidad $k$ |
| 2 | Probabilidad | $p[k] = h[k] / (L \cdot C)$ |
| 3 | CDF | $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$ |
| 4 | LUT | $\text{lut}[Look-Up Table (tabla de consulta)k] = \text{round}(\text{cdf}[k] \cdot (2^B-1))$ |
| 5 | Aplicación | $g[i,j] = \text{lut}[f[i,j]]$ |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $B$ (número de bits).
* Líneas siguientes: Elementos enteros de la matriz.

**Salida:**

* Matriz equalizada en $L$ líneas y $C$ columnas.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 5<br>5<br>3<br>3 4 2 3 4<br>4 3 3 4 3<br>2 3 4 3 2<br>3 4 3 2 3<br>4 3 2 3 4 | 5 7 1 5 7<br>7 5 5 7 5<br>1 5 7 5 1<br>5 7 5 1 5<br>7 5 1 5 7 | Ejemplo 3 bits del capítulo |
| 1<br>4<br>3<br>0 0 7 7 | 0 0 7 7 | Histograma bimodal extremo |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0304-equalizacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 Simulador EP03_04: Ecualización de Histograma</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">lut[k] = round(cdf[k] · (L − 1))</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Elija la profundidad de bits (B) y genere imágenes para analizar la dispersión dinámica del histograma y la tabla de remapeo (LUT) en tiempo real.</p>

    <!-- Controle de Bits B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#16a085;">Profundidad de Bits (B)</label>
        <span id="sim_ep0304_vl_bits" style="font-family:monospace;font-size:12px;font-weight:700;color:#16a085;">3 bits → 8 niveles</span>
      </div>
      <input type="range" id="sim_ep0304_sl_bits" min="1" max="8" step="1" value="3" style="width:100%;cursor:pointer;">
      <div style="display:flex;justify-content:space-between;margin-top:6px;font-size:10px;color:#8a8371;font-family:monospace;">
        <span>1 bit (2 niveles)</span>
        <span>4 bits (16 niveles)</span>
        <span>8 bits (256 niveles)</span>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Equalizada -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:16px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original</span>
        <div id="sim_ep0304_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0304_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Resultado Equalizado -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Ecualizado</span>
        <div id="sim_ep0304_grid_eq" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0304_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Nuevo Muestreo</button>
      </div>

    </div>

    <!-- Histogramas Lado a Lado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:16px;margin-bottom:16px;">
      
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Histograma Original</span>
        <canvas id="sim_ep0304_hist_orig" style="width:100%;height:80px;display:block;" width="340" height="80"></canvas>
      </div>

      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Histograma Ecualizado</span>
        <canvas id="sim_ep0304_hist_eq" style="width:100%;height:80px;display:block;" width="340" height="80"></canvas>
      </div>

    </div>

    <!-- Tabela LUT -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px 14px;margin-bottom:14px;overflow-x:auto;">
      <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:6px;">LUT (Tabla de Remapeo k → v)</span>
      <div id="sim_ep0304_lut_table" style="font-family:monospace;font-size:11px;color:#26241d;white-space:nowrap;"></div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0304_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      lut[k] = round(cdf[k] · 7) | B=3, niveles=8
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0304(root){
    if (!root || root.dataset.simEp0304Init) return;
    root.dataset.simEp0304Init = "1";

    var slBits     = root.querySelector('#sim_ep0304_sl_bits');
    var vlBits     = root.querySelector('#sim_ep0304_vl_bits');
    var debugDiv   = root.querySelector('#sim_ep0304_debug');
    var gridOrig   = root.querySelector('#sim_ep0304_grid_orig');
    var gridEq     = root.querySelector('#sim_ep0304_grid_eq');
    var lutTable   = root.querySelector('#sim_ep0304_lut_table');
    var histOrig   = root.querySelector('#sim_ep0304_hist_orig');
    var histEq     = root.querySelector('#sim_ep0304_hist_eq');

    var btnNew     = root.querySelector('#sim_ep0304_btnNew');
    var btnReset   = root.querySelector('#sim_ep0304_btnReset');

    var ROWS = 4, COLS = 4;
    var pixels = [];

    function generatePixels() {
      var b = parseInt(slBits.value) || 3;
      var levels = Math.pow(2, b);
      var lo = Math.floor(levels * 0.2);
      var hi = Math.floor(levels * 0.5);
      pixels = Array.from({ length: ROWS * COLS }, function(){
        return lo + Math.floor(Math.random() * (hi - lo + 1));
      });
    }

    function computeEqualization(pixArr, b) {
      var levels = Math.pow(2, b);
      var N = pixArr.length;
      var h = new Array(levels).fill(0);
      pixArr.forEach(function(p){ h[p]++; });

      var cdf = new Array(levels).fill(0);
      cdf[0] = h[0] / N;
      for (var k = 1; k < levels; k++) {
        cdf[k] = cdf[k - 1] + h[k] / N;
      }

      var lut = cdf.map(function(c){ return Math.round(c * (levels - 1)); });
      var result = pixArr.map(function(p){ return lut[p]; });
      return { h: h, cdf: cdf, lut: lut, result: result };
    }

    function drawHistogram(canvas, counts, levels, color) {
      var ctx = canvas.getContext('2d');
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0, 0, W, H);

      var maxVal = Math.max.apply(null, counts.concat([1]));
      var barW = W / levels;

      counts.forEach(function(c, i){
        var barH = (c / maxVal) * (H - 6);
        ctx.fillStyle = color;
        ctx.fillRect(i * barW + 1, H - barH, barW - 2, barH);
      });
    }

    function toGray(val, levels) {
      return Math.round((val / (levels - 1)) * 255);
    }

    function render() {
      var b = parseInt(slBits.value) || 3;
      var levels = Math.pow(2, b);
      vlBits.textContent = b + ' bit' + (b > 1 ? 's' : '') + ' → ' + levels + ' níveis';

      var eqData = computeEqualization(pixels, b);
      var hOrig = eqData.h;
      var lut = eqData.lut;
      var result = eqData.result;

      var hEq = new Array(levels).fill(0);
      result.forEach(function(p){ hEq[p]++; });

      lutTable.innerHTML = lut.map(function(v, k){
        return '<span style="display:inline-block;margin-right:10px;color:#8a8371;">' + k + ' → <b style="color:#16a085;">' + v + '</b></span>';
      }).join('');

      debugDiv.innerHTML = '<b>lut[k] = round(cdf[k] · ' + (levels - 1) + ')</b> &nbsp;|&nbsp; B = ' + b + ', níveis = ' + levels;

      gridOrig.innerHTML = '';
      gridEq.innerHTML = '';

      pixels.forEach(function(p, i) {
        var grayO = toGray(p, levels);
        var fgO = grayO > 128 ? '#000000' : '#ffffff';
        var cellO = document.createElement('div');
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + grayO + ',' + grayO + ',' + grayO + ');color:' + fgO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gridOrig.appendChild(cellO);

        var r = result[i];
        var grayR = toGray(r, levels);
        var fgR = grayR > 128 ? '#000000' : '#ffffff';
        var cellR = document.createElement('div');
        cellR.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + grayR + ',' + grayR + ',' + grayR + ');color:' + fgR + ';box-sizing:border-box;';
        cellR.textContent = r;
        gridEq.appendChild(cellR);
      });

      drawHistogram(histOrig, hOrig, levels, '#95a5a6');
      drawHistogram(histEq, hEq, levels, '#16a085');
    }

    slBits.addEventListener('input', function(){
      generatePixels();
      render();
    });

    btnNew.addEventListener('click', function(){
      generatePixels();
      render();
    });

    btnReset.addEventListener('click', function(){
      generatePixels();
      render();
    });

    generatePixels();
    render();
  }

  function tryInitSimEP0304(){
    var root = document.getElementById('sim-ep0304-equalizacao');
    if (root) initSimEP0304(root); else setTimeout(tryInitSimEP0304, 200);
  }
  tryInitSimEP0304();
})();
</script>
</div>
""")

**Figura 3.4:** Simulador EP03_04: Ecualización de Histograma (Niveles L = 2^B)


<figure id="fig-03-sim-ep0304-equalizacao">
  <img src="imagens/fig-03-sim-ep0304-equalizacao.png" alt=" Simulador EP03_04: Ecualización de Histograma (Niveles L = 2^B) " style="max-width:80%" />
  <figcaption><strong>Figura 3.4:</strong>  Simulador EP03_04: Ecualización de Histograma (Niveles L = 2^B) </figcaption>
</figure>

In [ ]:
%%writefile EP03_04.py
# Código Python

In [ ]:
TestSuite("EP03_04.py").run()

### EP03_05 🔲 Aplicación de Máscara AND Binaria

En sistemas de inspección industrial por visión por computadora, es necesario aislar regiones de interés (ROI) en imágenes de piezas para verificar defectos de fabricación. La operación **AND bit a bit con una máscara binaria** es el mecanismo fundamental para recortar exactamente el área de inspección, poniendo a cero todos los píxeles fuera de ella.

Ver en [Figura 3.5](#fig-03-sim-ep0305-mascara) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer la matriz de píxeles $f$ (valores $\in [0, 255]$).
3. **Máscara:** Leer la matriz binaria $m$ (valores: solo 0 o 255).
4. **Mapeo:** Para cada píxel $(i,j)$, aplicar el AND bit a bit:

$$
g(i,j) = f(i,j) \;\text{AND}\; m(i,j)
$$

donde $255 =$ `11111111` y $0 =$ `00000000` en binario.

5. **Salida:** Mostrar la matriz resultante $L \times C$.

#### 📌 Restricciones Computacionales

* **AND con 255:** $p \; \text{AND} \; 255 = p$ (todos los bits preservados).
* **AND con 0:** $p \; \text{AND} \; 0 = 0$ (todos los bits puestos a cero).
* **Máscara:** Los únicos valores posibles en la máscara son 0 y 255.
* **Implementación:** En Python, el AND bit a bit entre enteros usa el operador `&`.

#### 🧠 Fundamentación Teórica

| Píxel $f$ | Máscara $m$ | Resultado $f$ AND $m$ |
|:---------:|:-----------:|:---------------------:|
| cualquier $v$ | 255 (`11111111`) | $v$ (preservado) |
| cualquier $v$ | 0 (`00000000`) | 0 (puesto a cero) |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos de $f$ ($L$ líneas).
* Líneas siguientes: Elementos de $m$ ($L$ líneas con valores 0 o 255).

**Salida:**

* Matriz resultante $L \times C$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 2<br>3<br>100 150 200<br>50 80 120<br>255 255 0<br>0 255 255 | 100 150 0<br>0 80 120 | La máscara selecciona la región |
| 1<br>4<br>10 20 30 40<br>255 0 255 0 | 10 0 30 0 | Alternado preservado/puesto a cero |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0305-mascara" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⬛ Simulador EP03_05: Máscara AND Binaria</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f AND m</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en las celdas de la <b>Máscara m</b> para alternar entre transparente (255) y bloqueante (0), aplicando la operación lógica píxel a píxel.</p>

    <!-- Três Colunas Principais: Imagem f, Máscara m, Resultado g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:16px;">
      
      <!-- Imagem f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen f (0–255)</span>
        <div id="sim_ep0305_grid_f" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0305_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Máscara m -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Máscara m (Clic para Alternar)</span>
        <div id="sim_ep0305_grid_mask" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0305_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↺ Reiniciar Máscara</button>
      </div>

      <!-- Resultado g -->
      <div style="background:#fafaf7;border:2px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado g = f AND m</span>
        <div id="sim_ep0305_grid_result" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;height:26px;display:flex;align-items:center;justify-content:center;">
          <span id="sim_ep0305_pct" style="font-size:11px;font-weight:700;color:#26241d;font-family:monospace;">—</span>
        </div>
      </div>

    </div>

    <!-- Estatísticas de Preservação -->
    <div style="display:grid;grid-template-columns:repeat(3, minmax(0, 1fr));gap:10px;margin-bottom:14px;text-align:center;">
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_preserved" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>conservados
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_zeroed" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>puestos a cero
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_ratio" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>visible
      </div>
    </div>

    <!-- Legenda e Debug -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:24px;height:24px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;color:#2980b9;">255</div>
        <span style="font-size:10.5px;color:#5e5a4a;">Transparente (conservado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:24px;height:24px;background:#26241d;border:1.5px solid #8a8371;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;color:#7ee7c6;">0</div>
        <span style="font-size:10.5px;color:#5e5a4a;">Bloqueante (puesto a cero)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0305_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(i,j) = f(i,j) &amp; m(i,j)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0305(root){
    if (!root || root.dataset.simEp0305Init) return;
    root.dataset.simEp0305Init = "1";

    var gridF      = root.querySelector('#sim_ep0305_grid_f');
    var gridMask   = root.querySelector('#sim_ep0305_grid_mask');
    var gridResult = root.querySelector('#sim_ep0305_grid_result');
    var debugDiv   = root.querySelector('#sim_ep0305_debug');
    var pctSpan    = root.querySelector('#sim_ep0305_pct');
    var statPres   = root.querySelector('#sim_ep0305_stat_preserved');
    var statZero   = root.querySelector('#sim_ep0305_stat_zeroed');
    var statRatio  = root.querySelector('#sim_ep0305_stat_ratio');

    var btnNew     = root.querySelector('#sim_ep0305_btnNew');
    var btnReset   = root.querySelector('#sim_ep0305_btnReset');

    var N = 16;
    var pixels = [];
    var mask = [];

    function generatePixels() {
      pixels = Array.from({ length: N }, function(){ return Math.floor(Math.random() * 256); });
    }

    function resetMask() {
      mask = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / 4), c = i % 4;
        return (r + c) % 2 === 0 ? 255 : 0;
      });
    }

    function render() {
      gridF.innerHTML = '';
      gridMask.innerHTML = '';
      gridResult.innerHTML = '';

      var preserved = 0, zeroed = 0;

      for (var i = 0; i < N; i++) {
        var p = pixels[i];
        var m = mask[i];
        var res = p & m;

        // Célula F
        var cellF = document.createElement('div');
        var fgF = p > 128 ? '#000000' : '#ffffff';
        cellF.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgF + ';box-sizing:border-box;';
        cellF.textContent = p;
        gridF.appendChild(cellF);

        // Célula Máscara M (interativa)
        var cellM = document.createElement('div');
        cellM.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;user-select:none;box-sizing:border-box;transition:all 0.15s ease;';
        if (m === 255) {
          cellM.style.background = '#ebf4fd';
          cellM.style.border = '2px solid #2980b9';
          cellM.style.color = '#2980b9';
        } else {
          cellM.style.background = '#26241d';
          cellM.style.border = '2px solid #8a8371';
          cellM.style.color = '#7ee7c6';
        }
        cellM.textContent = m;
        cellM.title = m === 255 ? 'Clique para bloquear (0)' : 'Clique para passar (255)';

        (function(idx){
          cellM.addEventListener('click', function(){
            mask[idx] = mask[idx] === 255 ? 0 : 255;
            render();
          });
        })(i);

        gridMask.appendChild(cellM);

        // Célula Resultado G
        var cellR = document.createElement('div');
        var fgR = res > 128 ? '#000000' : '#ffffff';
        var borderStyle = m === 255 ? '2px solid #27ae60' : '1px solid #e4dcc8';
        cellR.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgR + ';border:' + borderStyle + ';box-sizing:border-box;';
        cellR.textContent = res;
        gridResult.appendChild(cellR);

        if (m === 255) preserved++; else zeroed++;
      }

      var ratioPct = Math.round((preserved / N) * 100);
      statPres.textContent = preserved;
      statZero.textContent = zeroed;
      statRatio.textContent = ratioPct + '%';
      pctSpan.textContent = ratioPct + '% visível';
      debugDiv.innerHTML = 'g(i,j) = f(i,j) &amp; m(i,j) &nbsp;|&nbsp; <b>' + preserved + '</b> preservados · <b>' + zeroed + '</b> zerados';
    }

    btnNew.addEventListener('click', function(){
      generatePixels();
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMask();
      render();
    });

    generatePixels();
    resetMask();
    render();
  }

  function tryInitSimEP0305(){
    var root = document.getElementById('sim-ep0305-mascara');
    if (root) initSimEP0305(root); else setTimeout(tryInitSimEP0305, 200);
  }
  tryInitSimEP0305();
})();
</script>
</div>
""")

**Figura 3.5:** Simulador EP03_05: Aplicación de Máscara AND Binaria


<figure id="fig-03-sim-ep0305-mascara">
  <img src="imagens/fig-03-sim-ep0305-mascara.png" alt=" Simulador EP03_05: Aplicación de Máscara AND Binaria " style="max-width:80%" />
  <figcaption><strong>Figura 3.5:</strong>  Simulador EP03_05: Aplicación de Máscara AND Binaria </figcaption>
</figure>

In [ ]:
%%writefile EP03_05.py
# Código de Python

In [ ]:
TestSuite("EP03_05.py").run()

### EP03_06 🌫️ Filtro de Media con *Kernel* N×N


En cámaras de vehículos autónomos, las imágenes capturadas bajo lluvia o niebla presentan ruido gaussiano. El **filtro de media** se utiliza ampliamente para su reducción en tiempo real, implementándose directamente en el **ISP** (*Image Signal Processor*) de los sensores **CMOS** (*Complementary Metal-Oxide-Semiconductor*).

Los sensores CMOS son los sensores de imagen utilizados en la mayoría de las cámaras modernas (*smartphones*, *webcams*, cámaras automotrices, etc.). Convierten la luz en señales eléctricas, y el ISP procesa estas señales en tiempo real — aplicando operaciones como reducción de ruido, balance de blancos y otros ajustes de imagen.

Ver en la [Figura 3.6](#fig-03-sim-ep0306-media) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas), $C$ (columnas) y $N$ (tamaño del *kernel*, siempre impar).
2. **Datos:** Leer la matriz de píxeles $f$.
3. **Filtro de Media:** Para cada píxel $(i,j)$ **interno** (sin bordes), calcular:

$$g(i,j) = \text{round}\left(\frac{1}{N^2} \sum_{s=-(r)}^{r} \sum_{t=-(r)}^{r} f(i+s,\, j+t)\right), \quad r = \lfloor N/2 \rfloor$$

4. **Tratamiento de Bordes:** Los píxeles en el borde (donde la ventana $N \times N$ sobrepasa los límites) deben **copiarse directamente** del original sin modificación.
5. **Salida:** Mostrar la matriz resultante $L \times C$.

#### 📌 Restricciones Computacionales

* **Radio:** $r = \lfloor N/2 \rfloor$ (mitad del *kernel*, entero).
* **Píxeles internos:** $(i,j)$ con $r \le i < L-r$ y $r \le j < C-r$.
* **Redondeo:** Usar redondeo matemático antes de convertir a entero.
* **Sin *clipping*:** El promedio de valores $\in [0,255]$ permanece en $[0,255]$.

#### 🧠 Fundamentación Teórica

| Tamaño $N$ | Coeficiente | Píxeles en la ventana | Efecto |
|:-----------:|:-----------:|:---------------------:|:------:|
| 3 | $1/9 \approx 0.111$ | 9 | Suave |
| 5 | $1/25 = 0.04$ | 25 | Medio |
| 7 | $1/49 \approx 0.020$ | 49 | Fuerte |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $N$ (impar, $N \ge 3$).
* Líneas siguientes: Elementos de la matriz original.

**Salida:**

* Matriz filtrada $L \times C$.

#### 📌 Ejemplos
| Entrada | Salida | Observación |
|---------|--------|-------------|
| 3<br>3<br>3<br>10 20 30<br>40 50 60<br>70 80 90 | 10 20 30<br>40 50 60<br>70 80 90 | Solo borde (3×3 = borde total) |
| 5<br>5<br>3<br>0 0 0 0 0<br>0 0 0 0 0<br>0 0 100 0 0<br>0 0 0 0 0<br>0 0 0 0 0 | 0 0 0 0 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 0 0 0 0 | Píxel aislado: todos los 9 píxeles internos cuya ventana 3×3 incluye el valor 100 reciben round(100/9)=11 |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0306-media" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🔲 Simulador EP03_06: Filtro de Media con Kernel N×N</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = Media(Vecinos)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Seleccione el tamaño del kernel y pase el mouse sobre los píxeles del resultado para inspeccionar la vecindad y el cálculo de la media aritmética.</p>

    <!-- Barra de Controles / Seleção de Kernel -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Tamaño del kernel:</span>
        <button id="sim_ep0306_btn_k3" style="padding:5px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">3 × 3 (9 vecinos)</button>
        <button id="sim_ep0306_btn_k5" style="padding:5px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">5 × 5 (25 vecinos)</button>
      </div>
      <button id="sim_ep0306_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
    </div>

    <!-- Comparativo Lado a Lado: Original vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original (7x7) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagen Original f (7×7)</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Con ruido sal y pimienta</span>
        <div id="sim_ep0306_grid_f" style="display:grid;gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Suavizado -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Resultado g (Filtro Suavizado)</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Pase el mouse para inspeccionar</span>
        <div id="sim_ep0306_grid_result" style="display:grid;gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Ventana del Kernel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Copiado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Inspeccionado</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0306_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pase el mouse sobre un píxel interno del resultado para ver el cálculo de la media.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0306(root){
    if (!root || root.dataset.simEp0306Init) return;
    root.dataset.simEp0306Init = "1";

    var ROWS = 7, COLS = 7, N = 49;
    var pixels = [], result = [], kSize = 3, radius = 1;

    var gridF   = root.querySelector('#sim_ep0306_grid_f');
    var gridRes = root.querySelector('#sim_ep0306_grid_result');
    var debug   = root.querySelector('#sim_ep0306_debug');
    var btnK3   = root.querySelector('#sim_ep0306_btn_k3');
    var btnK5   = root.querySelector('#sim_ep0306_btn_k5');
    var btnNew  = root.querySelector('#sim_ep0306_btnNew');

    function generatePixels() {
      pixels = Array.from({ length: N }, function(){
        var v = 60 + Math.floor(Math.random() * 60);
        if (Math.random() > 0.82) v = Math.random() > 0.5 ? 255 : 0;
        return v;
      });
    }

    function calculateFilter() {
      result = pixels.slice();
      radius = Math.floor(kSize / 2);
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          if (r >= radius && r < ROWS - radius && c >= radius && c < COLS - radius) {
            var sum = 0, cnt = 0;
            for (var s = -radius; s <= radius; s++) {
              for (var t = -radius; t <= radius; t++) {
                sum += pixels[(r + s) * COLS + (c + t)];
                cnt++;
              }
            }
            result[r * COLS + c] = Math.round(sum / cnt);
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function highlightKernel(tr, tc, on) {
      var cells = gridF.children;
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var idx = r * COLS + c;
          if (!cells[idx]) continue;
          var p = pixels[idx];
          if (on && Math.abs(r - tr) <= radius && Math.abs(c - tc) <= radius) {
            cells[idx].style.background = '#fef5e7';
            cells[idx].style.color = '#b9770e';
            cells[idx].style.boxShadow = '0 0 0 2px #b9770e inset';
          } else {
            cells[idx].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
            cells[idx].style.color = textColor(p);
            cells[idx].style.boxShadow = 'none';
          }
        }
      }
    }

    function render() {
      var cols = 'repeat(' + COLS + ', 42px)';
      gridF.style.gridTemplateColumns = cols;
      gridRes.style.gridTemplateColumns = cols;
      gridF.innerHTML = '';
      gridRes.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], res = result[i];
        var isBorder = r < radius || r >= ROWS - radius || c < radius || c >= COLS - radius;

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '2px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel (' + row + ',' + col + ') é <b>borda</b>: valor herdado do original sem cálculo &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              highlightKernel(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var neighbors = [];
              for (var s = -radius; s <= radius; s++) {
                for (var t = -radius; t <= radius; t++) {
                  neighbors.push(pixels[(row + s) * COLS + (col + t)]);
                }
              }
              var kTotal = kSize * kSize;
              debug.innerHTML = 'Pixel (' + row + ',' + col + '): round( (' + neighbors.join(' + ') + ') / ' + kTotal + ' ) &nbsp;=&nbsp; <b>' + val + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlightKernel(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + val + ',' + val + ',' + val + ')';
              cr.style.color = textColor(val);
              resetDebug();
            });
          })(r, c, res);
        }
        gridRes.appendChild(cr);
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno do resultado para ver o cálculo da média.';
    }

    function setKernel(k) {
      kSize = k;
      if (k === 3) {
        btnK3.style.background = '#ebf4fd';
        btnK3.style.borderColor = '#2980b9';
        btnK3.style.color = '#2980b9';
        btnK3.style.fontWeight = '700';

        btnK5.style.background = '#f1ead7';
        btnK5.style.borderColor = '#e4dcc8';
        btnK5.style.color = '#5e5a4a';
        btnK5.style.fontWeight = '600';
      } else {
        btnK5.style.background = '#ebf4fd';
        btnK5.style.borderColor = '#2980b9';
        btnK5.style.color = '#2980b9';
        btnK5.style.fontWeight = '700';

        btnK3.style.background = '#f1ead7';
        btnK3.style.borderColor = '#e4dcc8';
        btnK3.style.color = '#5e5a4a';
        btnK3.style.fontWeight = '600';
      }
      calculateFilter();
      render();
    }

    btnK3.addEventListener('click', function(){ setKernel(3); });
    btnK5.addEventListener('click', function(){ setKernel(5); });

    btnNew.addEventListener('click', function(){
      generatePixels();
      calculateFilter();
      render();
    });

    generatePixels();
    calculateFilter();
    render();
  }

  function tryInitSimEP0306(){
    var root = document.getElementById('sim-ep0306-media');
    if (root) initSimEP0306(root); else setTimeout(tryInitSimEP0306, 200);
  }
  tryInitSimEP0306();
})();
</script>
</div>
""")

**Figura 3.6:** Simulador EP03_06: Filtro de Media con Kernel N×N


<figure id="fig-03-sim-ep0306-media">
  <img src="imagens/fig-03-sim-ep0306-media.png" alt=" Simulador EP03_06: Filtro de Media con Kernel N×N " style="max-width:80%" />
  <figcaption><strong>Figura 3.6:</strong>  Simulador EP03_06: Filtro de Media con Kernel N×N </figcaption>
</figure>

In [ ]:
%%writefile EP03_06.py
# Código Python

In [ ]:
TestSuite("EP03_06.py").run()

### EP03_07 🔍 Operador Laplaciano (w4) para Realce de Bordas

En tomografías de alta resolución, la nitidez de los bordes entre tejidos es crítica para el diagnóstico. El **operador Laplaciano** se utiliza ampliamente en *pipelines* de preprocesamiento de imágenes médicas para resaltar automáticamente los contornos anatómicos antes de la segmentación, evitando la intervención manual del radiólogo.

Ver en [Figura 3.7](#fig-03-sim-ep0307-laplaciano) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer la matriz de píxeles $f$.
3. **Laplaciano (w4):** Para cada píxel **interno** $(i,j)$ con $1 \le i < L-1$, $1 \le j < C-1$, calcular:

$$\nabla^2 f(i,j) = f(i-1,j) + f(i+1,j) + f(i,j-1) + f(i,j+1) - 4 \cdot f(i,j)$$

4. **Realce:** Calcular la imagen realzada:

$$g(i,j) = \text{clip}(f(i,j) - \nabla^2 f(i,j))$$

5. **Borde:** Los píxeles en el borde se copian directamente: $g(i,j) = f(i,j)$.
6. **Salida:** Mostrar la matriz realzada $L \times C$.

#### 📌 Restricciones Computacionales

* ***Kernel* w4:** $\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$ — solo vecinos-4.
* **Saturación:** $\text{clip}(x) = \max(0, \min(255, x))$ aplicado al resultado del realce.
* **Sin redondeo:** El Laplaciano utiliza solo sumas/restas de enteros.

#### 🧠 Fundamentación Teórica

| Región | $\nabla^2 f$ | Efecto del Realce |
|:------:|:------------:|:----------------:|
| Uniforme | $\approx 0$ | Sin alteración |
| Borde creciente | $< 0$ | Píxel aclarado |
| Borde decreciente | $> 0$ | Píxel oscurecido |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos de la matriz original.

**Salida:**

* Matriz realzada $L \times C$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|-------------|
| 3<br>3<br>0 0 0<br>0 100 0<br>0 0 0 | 0 0 0<br>0 255 0<br>0 0 0 | Pico aislado: lap=−400, g=100−(−400)=500 → clip=255 |
| 3<br>3<br>50 50 50<br>50 50 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Región uniforme: Laplaciano=0, sin alteración |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0307-laplaciano" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📐 Simulador EP03_07: Operador Laplaciano (w4)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ∓ ∇²f</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Seleccione la variante de realce y pase el mouse sobre los píxeles internos del resultado para inspeccionar la vecindad de 4 puntos y la ecuación del Laplaciano.</p>

    <!-- Barra de Controles / Seleção de Variante -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Variante:</span>
        <button id="sim_ep0307_btn_v1" style="padding:5px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">g = f − ∇²f (Realce Estándar)</button>
        <button id="sim_ep0307_btn_v2" style="padding:5px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">g = f + ∇²f (Invierte Señal)</button>
      </div>
      <button id="sim_ep0307_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuevo Escalón</button>
    </div>

    <!-- Grid Principal de Comparação (2 colunas + setas) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">① Imagen Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Escalón con ruido leve</span>
        <div id="sim_ep0307_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Laplaciano ∇²f -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">② Laplaciano ∇²f</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Bordes detectados (±128 shift)</span>
        <div id="sim_ep0307_grid_l" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Linha de Resultado g e Kernel w4 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Resultado g -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;" id="sim_ep0307_flabel">③ Resultado g = f − ∇²f</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Pase el mouse para inspeccionar</span>
        <div id="sim_ep0307_grid_res" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Estrutura do Kernel w4 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:6px;font-family:monospace;">Kernel w4 (4-Vecinos)</span>
        <div style="display:inline-grid;grid-template-columns:repeat(3, 30px);gap:2px;margin-bottom:6px;justify-content:center;">
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#b9770e;border:1px solid #b9770e;border-radius:4px;color:#ffffff;">−4</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
        </div>
        <span style="font-size:10px;color:#8a8371;display:block;font-family:monospace;">∇²f = T + B + L + R − 4·f</span>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">4-Vecinos del Kernel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#faece7;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Copiado)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0307_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pase el mouse sobre un píxel interno del resultado para detallar la ecuación.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0307(root){
    if (!root || root.dataset.simEp0307Init) return;
    root.dataset.simEp0307Init = "1";

    var ROWS = 5, COLS = 5, N = 25;
    var pixels = [], lapV = [], lapA = [], result = [];
    var variant = 'subtract';

    var gridF   = root.querySelector('#sim_ep0307_grid_f');
    var gridL   = root.querySelector('#sim_ep0307_grid_l');
    var gridRes = root.querySelector('#sim_ep0307_grid_res');
    var debug   = root.querySelector('#sim_ep0307_debug');
    var fLabel  = root.querySelector('#sim_ep0307_flabel');
    var btnV1   = root.querySelector('#sim_ep0307_btn_v1');
    var btnV2   = root.querySelector('#sim_ep0307_btn_v2');
    var btnNew  = root.querySelector('#sim_ep0307_btnNew');

    function generate() {
      var sc = 2 + Math.floor(Math.random() * 2);
      var dark = 40 + Math.floor(Math.random() * 30);
      var light = 160 + Math.floor(Math.random() * 40);
      pixels = Array.from({ length: N }, function(_, i) {
        var c = i % COLS;
        var v = c < sc ? dark : light;
        v += Math.floor(Math.random() * 14) - 7;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      lapV = new Array(N).fill(0);
      lapA = new Array(N).fill(0);
      result = pixels.slice();

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= 1 && r < ROWS - 1 && c >= 1 && c < COLS - 1) {
            var t  = pixels[(r - 1) * COLS + c];
            var b  = pixels[(r + 1) * COLS + c];
            var l  = pixels[r * COLS + (c - 1)];
            var ri = pixels[r * COLS + (c + 1)];
            var f  = pixels[i];
            var lap = t + b + l + ri - 4 * f;

            lapV[i] = lap;
            lapA[i] = Math.max(0, Math.min(255, lap + 128));
            result[i] = Math.max(0, Math.min(255, variant === 'subtract' ? f - lap : f + lap));
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function highlightCross(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i];
        cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
        cells[i].style.color = textColor(p);
        cells[i].style.boxShadow = 'none';
      }
      if (on) {
        var ci = tr * COLS + tc;
        if (cells[ci]) {
          cells[ci].style.background = '#faece7';
          cells[ci].style.color = '#c0392b';
          cells[ci].style.boxShadow = '0 0 0 2px #c0392b inset';
        }
        var neighbors = [[tr - 1, tc], [tr + 1, tc], [tr, tc - 1], [tr, tc + 1]];
        neighbors.forEach(function(n) {
          var nr = n[0], nc = n[1];
          if (nr >= 0 && nr < ROWS && nc >= 0 && nc < COLS) {
            var ni = nr * COLS + nc;
            if (cells[ni]) {
              cells[ni].style.background = '#fef5e7';
              cells[ni].style.color = '#b9770e';
              cells[ni].style.boxShadow = '0 0 0 2px #b9770e inset';
            }
          }
        });
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno do resultado para detalhar a equação.';
    }

    function render() {
      gridF.innerHTML = '';
      gridL.innerHTML = '';
      gridRes.innerHTML = '';
      fLabel.textContent = variant === 'subtract' ? '③ Resultado g = f − ∇²f' : '③ Resultado g = f + ∇²f';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], lv = lapV[i], la = lapA[i], res = result[i];
        var isBorder = (r === 0 || r === ROWS - 1 || c === 0 || c === COLS - 1);

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Laplaciano l
        var cl = document.createElement('div');
        cl.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cl.style.background = '#fafaf7';
          cl.style.color = '#8a8371';
          cl.style.border = '1px solid #e4dcc8';
          cl.textContent = '—';
        } else {
          cl.style.background = 'rgb(' + la + ',' + la + ',' + la + ')';
          cl.style.color = textColor(la);
          cl.style.border = '1px solid #e4dcc8';
          cl.textContent = lv;
        }
        gridL.appendChild(cl);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): valor herdado sem cálculo &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, val, lapVal, origVal){
            var tVal = pixels[(row - 1) * COLS + col];
            var bVal = pixels[(row + 1) * COLS + col];
            var lVal = pixels[row * COLS + (col - 1)];
            var rVal = pixels[row * COLS + (col + 1)];

            cr.addEventListener('mouseenter', function(){
              highlightCross(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var sign = variant === 'subtract' ? '−' : '+';
              debug.innerHTML = '∇²f = (' + tVal + ' + ' + bVal + ' + ' + lVal + ' + ' + rVal + ') − 4·' + origVal + ' = <b>' + lapVal + '</b> &nbsp;|&nbsp; g = clip(' + origVal + ' ' + sign + ' ' + lapVal + ') = <b>' + val + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlightCross(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + val + ',' + val + ',' + val + ')';
              cr.style.color = textColor(val);
              resetDebug();
            });
          })(r, c, res, lv, p);
        }
        gridRes.appendChild(cr);
      }
    }

    function setVariant(v) {
      variant = v;
      if (v === 'subtract') {
        btnV1.style.background = '#fef5e7';
        btnV1.style.borderColor = '#b9770e';
        btnV1.style.color = '#b9770e';
        btnV1.style.fontWeight = '700';

        btnV2.style.background = '#f1ead7';
        btnV2.style.borderColor = '#e4dcc8';
        btnV2.style.color = '#5e5a4a';
        btnV2.style.fontWeight = '600';
      } else {
        btnV2.style.background = '#fef5e7';
        btnV2.style.borderColor = '#b9770e';
        btnV2.style.color = '#b9770e';
        btnV2.style.fontWeight = '700';

        btnV1.style.background = '#f1ead7';
        btnV1.style.borderColor = '#e4dcc8';
        btnV1.style.color = '#5e5a4a';
        btnV1.style.fontWeight = '600';
      }
      calculate();
      render();
    }

    btnV1.addEventListener('click', function(){ setVariant('subtract'); });
    btnV2.addEventListener('click', function(){ setVariant('add'); });

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0307(){
    var root = document.getElementById('sim-ep0307-laplaciano');
    if (root) initSimEP0307(root); else setTimeout(tryInitSimEP0307, 200);
  }
  tryInitSimEP0307();
})();
</script>
</div>
""")

**Figura 3.7:** Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas


<figure id="fig-03-sim-ep0307-laplaciano">
  <img src="imagens/fig-03-sim-ep0307-laplaciano.png" alt=" Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas " style="max-width:80%" />
  <figcaption><strong>Figura 3.7:</strong>  Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas </figcaption>
</figure>

In [ ]:
%%writefile EP03_07.py
# Código Python

In [ ]:
TestSuite("EP03_07.py").run()

### EP03_08 🧭 Gradiente de Sobel: Gx y Gy

En robots exploradores de Marte (como el Perseverance), la detección de obstáculos se realiza en tiempo real mediante cámaras estereoscópicas. El **operador de Sobel** calcula el gradiente direccional de la escena y se utiliza en el algoritmo de detección de bordes para identificar rocas, fisuras y desniveles del terreno que puedan comprometer la navegación.

Ver en [Figura 3.8](#fig-03-sim-ep0308-sobel) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer la matriz $f$.
3. **Gx y Gy:** Para cada píxel **interno** $(i,j)$ con $1 \le i < L-1$, $1 \le j < C-1$:

$$G_x(i,j) = [f(i-1,j+1) + 2f(i,j+1) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i,j-1) + f(i+1,j-1)]$$

$$G_y(i,j) = [f(i+1,j-1) + 2f(i+1,j) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i-1,j) + f(i-1,j+1)]$$

4. **Magnitud:** $|\nabla f(i,j)| = \text{clip}(\text{round}(\sqrt{G_x^2 + G_y^2}))$.
5. **Borde:** Los píxeles de borde reciben magnitud 0.
6. **Salida:** Mostrar la magnitud $L \times C$.

#### 📌 Restricciones Computacionales

* **Redondeo:** Aplicar `round` antes de convertir a entero.
* **Saturación:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Raíz cuadrada:** Usar $\sqrt{G_x^2 + G_y^2}$ (no la aproximación $|G_x| + |G_y|$).

#### 🧠 Fundamentación Teórica

| Operador | Detecta | Coeficientes diagonales |
|:--------:|:-------:|:----------------------:|
| $G_x$ | Bordes verticales | $\pm 1$ |
| $G_y$ | Bordes horizontales | $\pm 1$ |
| $|\nabla f|$ | Todos los bordes | Combinado |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos de la matriz.

**Salida:**

* Magnitud del gradiente, matriz $L \times C$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 0 0<br>0 0 0 | 0 0 0<br>0 0 0<br>0 0 0 | Imagen nula: gradiente cero |
| 3<br>3<br>0 0 255<br>0 0 255<br>0 0 255 | 0 0 0<br>0 255 0<br>0 0 0 | Borde vertical central: Gx alto |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0308-sobel" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧭 Simulador EP03_08: Gradiente de Sobel (Gx y Gy)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">|∇f| = √(Gx² + Gy²)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Analice la descomposición horizontal (Gx) y vertical (Gy) del operador de Sobel y pase el mouse sobre los píxeles de la magnitud para inspeccionar la vecindad 3×3.</p>

    <!-- Barra de Controles / Kernels Explicativos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      
      <div style="display:flex;align-items:center;gap:12px;flex-wrap:wrap;">
        <button id="sim_ep0308_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Escena</button>
        <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Kernels de Sobel:</span>
      </div>

      <!-- Representação Visual dos Kernels Gx e Gy -->
      <div style="display:flex;align-items:center;gap:16px;flex-wrap:wrap;">
        
        <!-- Kernel Gx -->
        <div style="display:flex;align-items:center;gap:6px;">
          <div style="display:inline-grid;grid-template-columns:repeat(3, 26px);gap:2px;background:#f1ead7;border-radius:6px;padding:4px;">
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+1</div>
          </div>
          <span style="font-size:11px;font-weight:700;color:#2980b9;font-family:monospace;">Gx</span>
        </div>

        <!-- Kernel Gy -->
        <div style="display:flex;align-items:center;gap:6px;">
          <div style="display:inline-grid;grid-template-columns:repeat(3, 26px);gap:2px;background:#f1ead7;border-radius:6px;padding:4px;">
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#633806;color:#fac775;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#854f0b;color:#fac775;border-radius:3px;">−2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#633806;color:#fac775;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ba7517;color:#faeeda;border-radius:3px;">+1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ef9f27;color:#412402;border-radius:3px;">+2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ba7517;color:#faeeda;border-radius:3px;">+1</div>
          </div>
          <span style="font-size:11px;font-weight:700;color:#b9770e;font-family:monospace;">Gy</span>
        </div>

      </div>

    </div>

    <!-- Comparativo em 2 Linhas (Original, Magnitude, Gx, Gy) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagen Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Matriz 5×5 píxeles</span>
        <div id="sim_ep0308_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Magnitude do Gradiente |∇f| -->
      <div style="background:#fafaf7;border:2px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Magnitud |∇f|</span>
        <span style="font-size:10px;color:#27ae60;display:block;margin-bottom:10px;">√(Gx² + Gy²)</span>
        <div id="sim_ep0308_grid_m" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Gradiente Horizontal Gx -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Gx — Gradiente Horizontal</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Azul = Negativo · Blanco = Cero · Azul Vivo = Positivo</span>
        <div id="sim_ep0308_grid_gx" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente Vertical Gy -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Gy — Gradiente Vertical</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Ámbar = Negativo · Blanco = Cero · Ámbar Vivo = Positivo</span>
        <div id="sim_ep0308_grid_gy" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Vecindad 3×3 Inspeccionada</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#26241d;border:1.5px solid #8a8371;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Forzado a 0)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0308_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pase el mouse sobre un píxel interno de la magnitud para ver la descomposición Gx y Gy.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0308(root){
    if (!root || root.dataset.simEp0308Init) return;
    root.dataset.simEp0308Init = "1";

    var ROWS = 5, COLS = 5, N = 25;
    var pixels = [], gxV = [], gyV = [], mgV = [];

    var gridF  = root.querySelector('#sim_ep0308_grid_f');
    var gridGx = root.querySelector('#sim_ep0308_grid_gx');
    var gridGy = root.querySelector('#sim_ep0308_grid_gy');
    var gridM  = root.querySelector('#sim_ep0308_grid_m');
    var debug  = root.querySelector('#sim_ep0308_debug');
    var btnNew = root.querySelector('#sim_ep0308_btnNew');

    function generate() {
      var block = Math.random() > 0.3;
      var bg = 30 + Math.floor(Math.random() * 30);
      var obj = 180 + Math.floor(Math.random() * 50);

      pixels = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var v = bg;
        if (block && r >= 2 && r <= 3 && c >= 2 && c <= 3) v = obj;
        else if (!block && r + c >= 4) v = obj - 40;
        v += Math.floor(Math.random() * 10) - 5;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      gxV = new Array(N).fill(0);
      gyV = new Array(N).fill(0);
      mgV = new Array(N).fill(0);

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= 1 && r < ROWS - 1 && c >= 1 && c < COLS - 1) {
            var p = [
              pixels[(r - 1) * COLS + (c - 1)], pixels[(r - 1) * COLS + c], pixels[(r - 1) * COLS + (c + 1)],
              pixels[r * COLS + (c - 1)],       pixels[r * COLS + c],       pixels[r * COLS + (c + 1)],
              pixels[(r + 1) * COLS + (c - 1)], pixels[(r + 1) * COLS + c], pixels[(r + 1) * COLS + (c + 1)]
            ];
            var gx = (p[2] + 2 * p[5] + p[8]) - (p[0] + 2 * p[3] + p[6]);
            var gy = (p[6] + 2 * p[7] + p[8]) - (p[0] + 2 * p[1] + p[2]);

            gxV[i] = gx;
            gyV[i] = gy;
            mgV[i] = Math.min(255, Math.round(Math.sqrt(gx * gx + gy * gy)));
          }
        }
      }
    }

    function textColor(g){ return g > 150 ? '#000000' : '#ffffff'; }

    function colorGxBetter(v) {
      var n = Math.max(-400, Math.min(400, v));
      if (Math.abs(n) < 15) return '#fafaf7';
      if (n > 0) {
        var t = Math.min(1, n / 350);
        var r = Math.round(4 + t * 20);
        var g = Math.round(44 + t * 71);
        var b = Math.round(83 + t * 89);
        return 'rgb(' + r + ',' + g + ',' + b + ')';
      } else {
        var t = Math.min(1, -n / 350);
        return 'rgb(' + Math.round(12 + t * 0) + ',' + Math.round(68 - t * 24) + ',' + Math.round(165 - t * 82) + ')';
      }
    }

    function colorGyBetter(v) {
      var n = Math.max(-400, Math.min(400, v));
      if (Math.abs(n) < 15) return '#fafaf7';
      if (n > 0) {
        var t = Math.min(1, n / 350);
        return 'rgb(' + Math.round(186 + t * 63) + ',' + Math.round(117 + t * 70) + ',' + Math.round(23 - t * 18) + ')';
      } else {
        var t = Math.min(1, -n / 350);
        return 'rgb(' + Math.round(99 + t * 0) + ',' + Math.round(56 - t * 20) + ',' + Math.round(11 - t * 5) + ')';
      }
    }

    function textSignedColor(bg) {
      if (bg === '#fafaf7') return '#26241d';
      var m = bg.match(/rgb\((\d+),(\d+),(\d+)\)/);
      if (!m) return '#ffffff';
      var lum = 0.299 * m[1] + 0.587 * m[2] + 0.114 * m[3];
      return lum > 140 ? '#000000' : '#ffffff';
    }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
          cells[i].style.color = textColor(p);
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno da magnitude para ver a decomposição Gx e Gy.';
    }

    function render() {
      gridF.innerHTML = '';
      gridGx.innerHTML = '';
      gridGy.innerHTML = '';
      gridM.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], gx = gxV[i], gy = gyV[i], mag = mgV[i];
        var isBorder = (r === 0 || r === ROWS - 1 || c === 0 || c === COLS - 1);

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Gx
        var cgx = document.createElement('div');
        cgx.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cgx.style.background = '#fafaf7';
          cgx.style.color = '#8a8371';
          cgx.style.border = '1px solid #e4dcc8';
          cgx.textContent = '—';
        } else {
          var bgGx = colorGxBetter(gx);
          cgx.style.background = bgGx;
          cgx.style.color = textSignedColor(bgGx);
          cgx.style.border = '1px solid #e4dcc8';
          cgx.textContent = (gx > 0 ? '+' : '') + gx;
        }
        gridGx.appendChild(cgx);

        // Célula Gy
        var cgy = document.createElement('div');
        cgy.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cgy.style.background = '#fafaf7';
          cgy.style.color = '#8a8371';
          cgy.style.border = '1px solid #e4dcc8';
          cgy.textContent = '—';
        } else {
          var bgGy = colorGyBetter(gy);
          cgy.style.background = bgGy;
          cgy.style.color = textSignedColor(bgGy);
          cgy.style.border = '1px solid #e4dcc8';
          cgy.textContent = (gy > 0 ? '+' : '') + gy;
        }
        gridGy.appendChild(cgy);

        // Célula Magnitude m
        var cm = document.createElement('div');
        cm.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';
        if (isBorder) {
          cm.style.background = '#26241d';
          cm.style.border = '1.5px solid #8a8371';
          cm.style.color = '#7ee7c6';
          cm.textContent = '0';

          (function(row, col){
            cm.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): sem vizinhança completa → forçado para <b>0</b>';
            });
            cm.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c);
        } else {
          cm.style.background = 'rgb(' + mag + ',' + mag + ',' + mag + ')';
          cm.style.color = textColor(mag);
          cm.style.border = '1px solid #e4dcc8';
          cm.style.cursor = 'pointer';
          cm.textContent = mag;

          (function(row, col, gxv, gyv, mgv){
            cm.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cm.style.transform = 'scale(1.12)';
              cm.style.boxShadow = '0 0 0 2px #27ae60 inset';
              cm.style.background = '#eafaf1';
              cm.style.color = '#27ae60';

              debug.innerHTML = 'Pixel (' + row + ',' + col + '): Gx = <b>' + gxv + '</b> &nbsp;|&nbsp; Gy = <b>' + gyv + '</b> &nbsp;|&nbsp; |∇f| = round(√(' + gxv + '² + ' + gyv + '²)) = <b>' + mgv + '</b>';
            });

            cm.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cm.style.transform = 'scale(1)';
              cm.style.boxShadow = 'none';
              cm.style.background = 'rgb(' + mgv + ',' + mgv + ',' + mgv + ')';
              cm.style.color = textColor(mgv);
              resetDebug();
            });
          })(r, c, gx, gy, mag);
        }
        gridM.appendChild(cm);
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0308(){
    var root = document.getElementById('sim-ep0308-sobel');
    if (root) initSimEP0308(root); else setTimeout(tryInitSimEP0308, 200);
  }
  tryInitSimEP0308();
})();
</script>
</div>
""")

**Figura 3.8:** Simulador EP03_08: Gradiente de Sobel (Gx y Gy)


<figure id="fig-03-sim-ep0308-sobel">
  <img src="imagens/fig-03-sim-ep0308-sobel.png" alt=" Simulador EP03_08: Gradiente de Sobel (Gx y Gy) " style="max-width:80%" />
  <figcaption><strong>Figura 3.8:</strong>  Simulador EP03_08: Gradiente de Sobel (Gx y Gy) </figcaption>
</figure>

In [ ]:
%%writefile EP03_08.py
# Código Python

In [ ]:
TestSuite("EP03_08.py").run()

### EP03_09 📡 Filtro de la Mediana 3×3

Las imágenes de radar de apertura sintética (SAR) utilizadas en monitoreo ambiental y militar sufren de un tipo específico de ruido llamado *speckle*, que posee características similares al ruido sal y pimienta. El **filtro de la mediana** es el método estándar para eliminar este ruido porque preserva los bordes de las estructuras mientras elimina los puntos espurios.

Ver en [Figura 3.9](#fig-03-sim-ep0309-mediana) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer la matriz de píxeles $f$.
3. **Filtro de la Mediana 3×3:** Para cada píxel **interno** $(i,j)$ con $1 \le i < L-1$, $1 \le j < C-1$:
   - Recolectar los 9 píxeles de la vecindad $3 \times 3$: $\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$.
   - Ordenar los 9 valores en orden creciente.
   - Asignar $g(i,j)$ al valor central (posición índice 4, considerando índice 0).

$$g(i,j) = \text{mediana}\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$$

4. **Borde:** Copiar directamente: $g(i,j) = f(i,j)$.
5. **Salida:** Mostrar la matriz filtrada $L \times C$.

#### 📌 Restricciones Computacionales

* **Ventana:** Siempre $3 \times 3 = 9$ elementos.
* **Mediana:** El elemento central de la secuencia ordenada (índice 4 de 0 a 8).
* **Sin recorte:** La mediana de valores en $[0, 255]$ permanece en $[0, 255]$.
* **No lineal:** El filtro de mediana no puede expresarse como convolución lineal.

#### 🧠 Fundamentación Teórica

| Ruido | Filtro de Media | Filtro de Mediana |
|:-----:|:---------------:|:-----------------:|
| Sal y pimienta (0 o 255) | Dispersa el ruido | Elimina sin distorsionar bordes |
| Gaussiano | Reduce eficazmente | Reduce parcialmente |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos de la matriz.

**Salida:**

* Matriz filtrada $L \times C$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 3<br>3<br>100 100 100<br>100 0 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | Punto negro eliminado: mediana de 8×100+1×0 = 100 |
| 3<br>3<br>50 50 50<br>50 255 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Punto blanco (sal) eliminado |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0309-mediana" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📉 Simulador EP03_09: Filtro de la Mediana 3×3</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = Mediana(Vecinos)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Inyecte ruido impulsivo (sal y pimienta) y pase el mouse sobre los píxeles internos del resultado para inspeccionar la ordenación del vector de vecindad y la eliminación del ruido.</p>

    <!-- Barra de Controles / Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      <button id="sim_ep0309_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Inyectar Ruido Impulsivo</button>
      <span style="font-size:11px;color:#8a8371;">Ruido de sal (255) y pimienta (0) — ~30% de los píxeles internos afectados</span>
    </div>

    <!-- Comparativo Lado a Lado: Original com Ruído vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem f com Ruído -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagen f — Con Ruido</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Sal (255) y pimienta (0) visibles</span>
        <div id="sim_ep0309_grid_f" style="display:grid;grid-template-columns:repeat(5, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado g sem Ruído -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Resultado g — Sin Ruido</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Pase el mouse para inspeccionar</span>
        <div id="sim_ep0309_grid_result" style="display:grid;grid-template-columns:repeat(5, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Vetor Ordenado -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:14px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:3px;">Vector de Vecindad 3×3 — Ordenado</span>
      <span style="font-size:10.5px;color:#8a8371;display:block;margin-bottom:8px;">Pase el mouse sobre un píxel interno del resultado para visualizar</span>
      <div id="sim_ep0309_vector" style="display:flex;flex-wrap:wrap;justify-content:center;gap:3px;min-height:32px;padding:4px 0;align-items:center;">
        <span style="font-size:11px;color:#8a8371;font-style:italic;">—</span>
      </div>
    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Ventana 3×3</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Copiado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Mediana</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#26241d;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Ruido (Eliminado)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0309_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pase el mouse sobre un píxel interno del resultado para ver el proceso de ordenación.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0309(root){
    if (!root || root.dataset.simEp0309Init) return;
    root.dataset.simEp0309Init = "1";

    var ROWS = 5, COLS = 5, N = 25, RADIUS = 1;
    var pixels = [], result = [];

    var gridF  = root.querySelector('#sim_ep0309_grid_f');
    var gridRes= root.querySelector('#sim_ep0309_grid_result');
    var vector = root.querySelector('#sim_ep0309_vector');
    var debug  = root.querySelector('#sim_ep0309_debug');
    var btnNew = root.querySelector('#sim_ep0309_btnNew');

    function generate() {
      pixels = Array.from({ length: N }, function(){
        var v = 110 + Math.floor(Math.random() * 30);
        var rnd = Math.random();
        if (rnd > 0.82) v = 255;
        else if (rnd < 0.18) v = 0;
        return v;
      });
      calculate();
    }

    function calculate() {
      result = pixels.slice();
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          if (r >= RADIUS && r < ROWS - RADIUS && c >= RADIUS && c < COLS - RADIUS) {
            var win = [];
            for (var s = -RADIUS; s <= RADIUS; s++) {
              for (var t = -RADIUS; t <= RADIUS; t++) {
                win.push(pixels[(r + s) * COLS + (c + t)]);
              }
            }
            win.sort(function(a, b){ return a - b; });
            result[r * COLS + c] = win[4];
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }
    function isNoise(v){ return v === 0 || v === 255; }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          if (isNoise(p)) {
            cells[i].style.background = p === 255 ? '#ffffff' : '#26241d';
            cells[i].style.color = p === 255 ? '#c0392b' : '#e74c3c';
            cells[i].style.border = '2px solid #c0392b';
          } else {
            cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
            cells[i].style.color = textColor(p);
            cells[i].style.border = '1px solid #e4dcc8';
          }
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function showVector(raw, sorted, median) {
      vector.innerHTML = '';

      var rawLabel = document.createElement('span');
      rawLabel.style.cssText = 'font-size:10px;color:#8a8371;margin-right:6px;font-family:monospace;';
      rawLabel.textContent = 'Bruto:';
      vector.appendChild(rawLabel);

      raw.forEach(function(v){
        var d = document.createElement('div');
        var noise = isNoise(v);
        d.style.cssText = 'display:inline-flex;align-items:center;justify-content:center;width:28px;height:24px;border-radius:4px;font-size:10px;font-weight:700;font-family:monospace;margin:1px;';
        d.style.background = noise ? '#26241d' : '#fafaf7';
        d.style.color = noise ? '#e74c3c' : '#26241d';
        d.style.border = noise ? '1.5px solid #c0392b' : '1px solid #e4dcc8';
        d.textContent = v;
        vector.appendChild(d);
      });

      var arr = document.createElement('span');
      arr.style.cssText = 'font-size:14px;margin:0 6px;color:#8a8371;font-weight:700;';
      arr.textContent = '→';
      vector.appendChild(arr);

      var sortLabel = document.createElement('span');
      sortLabel.style.cssText = 'font-size:10px;color:#8a8371;margin-right:6px;font-family:monospace;';
      sortLabel.textContent = 'Ordenado:';
      vector.appendChild(sortLabel);

      sorted.forEach(function(v, i){
        var d = document.createElement('div');
        var isMedian = (i === 4);
        var noise = isNoise(v);
        d.style.cssText = 'display:inline-flex;align-items:center;justify-content:center;width:28px;height:24px;border-radius:4px;font-size:10px;font-weight:700;font-family:monospace;margin:1px;';
        if (isMedian) {
          d.style.background = '#eafaf1';
          d.style.color = '#27ae60';
          d.style.border = '2px solid #27ae60';
        } else if (noise) {
          d.style.background = '#26241d';
          d.style.color = '#e74c3c';
          d.style.border = '1.5px solid #c0392b';
        } else {
          d.style.background = '#fafaf7';
          d.style.color = '#26241d';
          d.style.border = '1px solid #e4dcc8';
        }
        d.textContent = v;
        vector.appendChild(d);
      });

      var eq = document.createElement('span');
      eq.style.cssText = 'font-size:11px;margin-left:8px;font-family:monospace;color:#27ae60;font-weight:700;';
      eq.innerHTML = '→ mediana = <b>' + median + '</b>';
      vector.appendChild(eq);
    }

    function resetVector() {
      vector.innerHTML = '<span style="font-size:11px;color:#8a8371;font-style:italic;">—</span>';
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um píxel interno do resultado para ver o processo de ordenação.';
    }

    function render() {
      gridF.innerHTML = '';
      gridRes.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], res = result[i];
        var isBorder = (r < RADIUS || r >= ROWS - RADIUS || c < RADIUS || c >= COLS - RADIUS);
        var noise = isNoise(p);

        // Célula F com Ruído
        var cf = document.createElement('div');
        if (noise) {
          cf.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;background:' + (p === 255 ? '#ffffff' : '#26241d') + ';color:' + (p === 255 ? '#c0392b' : '#e74c3c') + ';border:2px solid #c0392b;box-sizing:border-box;';
        } else {
          cf.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        }
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): copiado sem filtro → <b>' + val + '</b>';
              resetVector();
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, resVal){
            cr.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var raw = [];
              for (var s = -RADIUS; s <= RADIUS; s++) {
                for (var t = -RADIUS; t <= RADIUS; t++) {
                  raw.push(pixels[(row + s) * COLS + (col + t)]);
                }
              }
              var sorted = raw.slice().sort(function(a, b){ return a - b; });
              showVector(raw, sorted, resVal);

              var noiseCount = raw.filter(isNoise).length;
              debug.innerHTML = 'Pixel (' + row + ',' + col + '): ' + noiseCount + ' vizinho(s) com ruído na janela &nbsp;→&nbsp; mediana = posição [4] = <b>' + resVal + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + resVal + ',' + resVal + ',' + resVal + ')';
              cr.style.color = textColor(resVal);
              resetDebug();
              resetVector();
            });
          })(r, c, res);
        }
        gridRes.appendChild(cr);
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0309(){
    var root = document.getElementById('sim-ep0309-mediana');
    if (root) initSimEP0309(root); else setTimeout(tryInitSimEP0309, 200);
  }
  tryInitSimEP0309();
})();
</script>
</div>
""")

**Figura 3.9:** Simulador EP03_09: Filtro de la Mediana 3×3


<figure id="fig-03-sim-ep0309-mediana">
  <img src="imagens/fig-03-sim-ep0309-mediana.png" alt=" Simulador EP03_09: Filtro de la Mediana 3×3 " style="max-width:80%" />
  <figcaption><strong>Figura 3.9:</strong>  Simulador EP03_09: Filtro de la Mediana 3×3 </figcaption>
</figure>

In [ ]:
%%writefile EP03_09.py
# Código Python

In [ ]:
TestSuite("EP03_09.py").run()

### EP03_10 ✨ *Unsharp Masking* (USM)

En los sistemas de digitalización de documentos históricos y obras de arte, la nitidez de las imágenes es fundamental para la lectura de textos manuscritos y detalles ornamentales. El ***Unsharp Masking* (USM)** es el algoritmo de realce de nitidez estándar utilizado en *escáneres* profesionales y software como Adobe Photoshop, controlado por el parámetro $k$ que determina la intensidad del realce.

Ver en la [Figura 3.10](#fig-03-sim-ep0310-unsharp) una simulación de este EP.


#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (líneas) y $C$ (columnas).
2. **Parámetro:** Leer el valor real $k$ (intensidad del realce, $k \ge 0$).
3. **Datos:** Leer la matriz de píxeles $f$.
4. **Suavizado:** Calcular $\bar{f}$ con el filtro de promedio $3\times3$ (solo píxeles internos; bordes mantenidos):

$$\bar{f}(i,j) = \frac{1}{9} \sum_{s=-1}^{1} \sum_{t=-1}^{1} f(i+s, j+t)$$

5. **Máscara de alta frecuencia:** $m(i,j) = f(i,j) - \bar{f}(i,j)$.
6. **Realce USM:** Para cada píxel interno:

$$g(i,j) = \text{clip}\left(\text{round}\left(f(i,j) + k \cdot m(i,j)\right)\right)$$

7. **Borde:** $g(i,j) = f(i,j)$ (copia directa).
8. **Salida:** Mostrar la matriz realzada $L \times C$.

#### 📌 Restricciones Computacionales

* **Redondeo:** Aplicar `round` antes del clip.
* **Saturación:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operaciones en float:** Calcular $\bar{f}$ y $m$ en punto flotante antes de redondear el resultado final.
* **$k = 0$:** Sin realce — la salida es idéntica a la entrada (excepto en los bordes).

#### 🧠 Fundamentación Teórica

| Etapa | Operación | Descripción |
|:-----:|:----------|:------------|
| 1 | $\bar{f} = f * \frac{1}{9}\mathbf{1}_{3\times3}$ | Suavizado (bajas frecuencias) |
| 2 | $m = f - \bar{f}$ | Máscara (altas frecuencias) |
| 3 | $g = \text{clip}(\text{round}(f + k \cdot m))$ | Realce ponderado |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Real $k$.
* Líneas siguientes: Elementos de la matriz original.

**Salida:**

* Matriz realzada $L \times C$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 3<br>3<br>0.0<br>100 100 100<br>100 100 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | k=0: sin realce |
| 3<br>3<br>1.0<br>50 50 50<br>50 200 50<br>50 50 50 | 50 50 50<br>50 255 50<br>50 50 50 | k=1: píxel central realzado y saturado |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0310-unsharp" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">✨ Simulador EP03_10: Enmascaramiento Unsharp (USM)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f + k · m</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajusta el factor de ganancia k, observa el pipeline completo de realce (desenfoque, máscara de alta frecuencia) y pasa el mouse sobre el resultado.</p>

    <!-- Barra de Controles / Slider de Ganho k -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      
      <div style="display:flex;align-items:center;gap:10px;flex:1;min-width:240px;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Factor de ganancia k:</span>
        <input id="sim_ep0310_k" type="range" min="0.0" max="3.0" step="0.5" value="1.0" style="flex:1;cursor:pointer;accent-color:#2980b9;">
        <span id="sim_ep0310_klabel" style="font-family:monospace;font-size:11px;font-weight:700;background:#26241d;color:#7ee7c6;padding:3px 8px;border-radius:6px;">k = 1.0</span>
      </div>

      <button id="sim_ep0310_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
    </div>

    <!-- Comparativo do Pipeline USM (Grid de 4 Cards) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- ① Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">① Imagen Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Matriz 5×5 píxeles</span>
        <div id="sim_ep0310_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ② Desfocado f_barra -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">② Desenfocado f̄</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Promedio 3 × 3</span>
        <div id="sim_ep0310_grid_b" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- ③ Máscara m -->
      <div style="background:#fafaf7;border:2px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">③ Máscara m</span>
        <span style="font-size:10px;color:#c0392b;display:block;margin-bottom:10px;">m = f − f̄ (Altas Frecuencias)</span>
        <div id="sim_ep0310_grid_m" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ④ Resultado g -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;" id="sim_ep0310_flabel">④ Resultado g = f + 1.0·m</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Pasa el mouse para inspeccionar</span>
        <div id="sim_ep0310_grid_res" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Vecindario 3×3</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Copiado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fbeaf0;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Máscara Positiva/Negativa</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0310_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pasa el mouse sobre un píxel interno del resultado para rastrear el pipeline completo.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0310(root){
    if (!root || root.dataset.simEp0310Init) return;
    root.dataset.simEp0310Init = "1";

    var ROWS = 5, COLS = 5, N = 25, RADIUS = 1;
    var pixels = [], blurred = [], mask = [], result = [];
    var gainK = 1.0;

    var gridF  = root.querySelector('#sim_ep0310_grid_f');
    var gridB  = root.querySelector('#sim_ep0310_grid_b');
    var gridM  = root.querySelector('#sim_ep0310_grid_m');
    var gridRes= root.querySelector('#sim_ep0310_grid_res');
    var debug  = root.querySelector('#sim_ep0310_debug');
    var fLabel = root.querySelector('#sim_ep0310_flabel');
    var sliderK= root.querySelector('#sim_ep0310_k');
    var kLabel = root.querySelector('#sim_ep0310_klabel');
    var btnNew = root.querySelector('#sim_ep0310_btnNew');

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function colorMask(v) {
      if (Math.abs(v) < 2) return '#fafaf7';
      if (v > 0) {
        var t = Math.min(1, v / 80);
        return 'rgb(' + Math.round(230 + t * 20) + ',' + Math.round(210 - t * 60) + ',' + Math.round(220 - t * 60) + ')';
      }
      var t = Math.min(1, -v / 80);
      return 'rgb(' + Math.round(210 - t * 60) + ',' + Math.round(220 - t * 40) + ',' + Math.round(230 + t * 20) + ')';
    }

    function generate() {
      pixels = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var v = (r >= 2 && c >= 2) ? 170 : 70;
        v += Math.floor(Math.random() * 16) - 8;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      blurred = new Array(N).fill(0);
      mask = new Array(N).fill(0);
      result = new Array(N).fill(0);

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= RADIUS && r < ROWS - RADIUS && c >= RADIUS && c < COLS - RADIUS) {
            var sum = 0;
            for (var dr = -RADIUS; dr <= RADIUS; dr++) {
              for (var dc = -RADIUS; dc <= RADIUS; dc++) {
                sum += pixels[(r + dr) * COLS + (c + dc)];
              }
            }
            var b = sum / 9;
            var m = pixels[i] - b;
            blurred[i] = b;
            mask[i] = m;
            result[i] = Math.max(0, Math.min(255, Math.round(pixels[i] + gainK * m)));
          } else {
            blurred[i] = pixels[i];
            mask[i] = 0;
            result[i] = pixels[i];
          }
        }
      }
    }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
          cells[i].style.color = textColor(p);
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um píxel interno do resultado para rastrear o pipeline completo.';
    }

    function render() {
      gridF.innerHTML = '';
      gridB.innerHTML = '';
      gridM.innerHTML = '';
      gridRes.innerHTML = '';
      fLabel.textContent = '④ Resultado g = f +' + gainK.toFixed(1) + '·m';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], b = blurred[i], m = mask[i], res = result[i];
        var isBorder = (r < RADIUS || r >= ROWS - RADIUS || c < RADIUS || c >= COLS - RADIUS);

        // Célula F
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Desfocado B
        var cb = document.createElement('div');
        cb.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cb.style.background = '#fafaf7';
          cb.style.color = '#8a8371';
          cb.style.border = '1px solid #e4dcc8';
          cb.textContent = '—';
        } else {
          var bv = Math.round(b);
          cb.style.background = 'rgb(' + bv + ',' + bv + ',' + bv + ')';
          cb.style.color = textColor(bv);
          cb.style.border = '1px solid #e4dcc8';
          cb.textContent = b.toFixed(1);
        }
        gridB.appendChild(cb);

        // Célula Máscara M
        var cm = document.createElement('div');
        cm.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cm.style.background = '#fafaf7';
          cm.style.color = '#8a8371';
          cm.style.border = '1px solid #e4dcc8';
          cm.textContent = '0';
        } else {
          var sg = m >= 0 ? '+' : '';
          cm.style.background = colorMask(m);
          cm.style.color = '#26241d';
          cm.style.border = '1px solid #e4dcc8';
          cm.textContent = sg + m.toFixed(1);
        }
        gridM.appendChild(cm);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): copiado sem alteração &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, pVal, bVal, mVal, resVal){
            cr.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var sg = mVal >= 0 ? '+' : '';
              var raw = pVal + gainK * mVal;
              debug.innerHTML = '① f=' + pVal + ' &nbsp;→&nbsp; ② f̄=' + bVal.toFixed(1) + ' &nbsp;→&nbsp; ③ m=f−f̄=' + sg + mVal.toFixed(1) + ' &nbsp;→&nbsp; ④ g = clip(' + pVal + ' + ' + gainK.toFixed(1) + '·(' + sg + mVal.toFixed(1) + ')) = clip(' + raw.toFixed(1) + ') = <b>' + resVal + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + resVal + ',' + resVal + ',' + resVal + ')';
              cr.style.color = textColor(resVal);
              resetDebug();
            });
          })(r, c, p, b, m, res);
        }
        gridRes.appendChild(cr);
      }
    }

    sliderK.addEventListener('input', function(e){
      gainK = parseFloat(e.target.value) || 0;
      kLabel.textContent = 'k = ' + gainK.toFixed(1);
      calculate();
      render();
    });

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0310(){
    var root = document.getElementById('sim-ep0310-unsharp');
    if (root) initSimEP0310(root); else setTimeout(tryInitSimEP0310, 200);
  }
  tryInitSimEP0310();
})();
</script>
</div>
""")

**Figura 3.10:** Simulador EP03_10: *Máscara de enfoque* (USM)


<figure id="fig-03-sim-ep0310-unsharp">
  <img src="imagens/fig-03-sim-ep0310-unsharp.png" alt=" Simulador EP03_10: *Máscara de enfoque* (USM) " style="max-width:80%" />
  <figcaption><strong>Figura 3.10:</strong>  Simulador EP03_10: *Máscara de enfoque* (USM) </figcaption>
</figure>

In [ ]:
%%writefile EP03_10.py
# Código Python

In [ ]:
TestSuite("EP03_10.py").run()